# Perform organoid-level quality control

In [1]:
import os
import pathlib
import sys

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    image_based_profiles_subparent_name = "image_based_profiles"
    patient = "NF0014_T1"

In [3]:
print(f"Processing patient: {patient}")
print("This should not read none....")

Processing patient: NF0014_T1
This should not read none....


## Load in all the organoid profiles and concat together

In [4]:
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles"
    / "organoid_anno.parquet"
).resolve(strict=True)

output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

orig_organoid_profiles_df = pd.read_parquet(organoid_file)

# Print the shape and head of the combined organoid profiles DataFrame
print(orig_organoid_profiles_df.shape)
orig_organoid_profiles_df.head()

(18, 891)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Organoid_CenterX,Metadata_Location_Organoid_CenterY,Metadata_Location_Organoid_CenterZ,...,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_EquivalentDiameter,Organoid_NoChannel_AreaSizeShape_EulerNumber,Organoid_NoChannel_AreaSizeShape_Extent,Organoid_NoChannel_AreaSizeShape_SurfaceArea,Organoid_NoChannel_AreaSizeShape_Volume
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.969913,393.226576,3.055781,...,1.766927e-284,1.766927e-284,0.000000e+00,1.766927e-284,72576.0,46.946452,1,0.746473,184.733753,54176.0
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.988177,254.361071,3.051103,...,1.766927e-284,1.766927e-284,0.000000e+00,1.766927e-284,111804.0,54.388652,1,0.753470,231.971359,84241.0
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,470.988847,423.997957,3.000954,...,1.766927e-284,1.766927e-284,0.000000e+00,1.766927e-284,87230.0,50.408459,1,0.768852,148.222080,67067.0
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,416.447720,1092.972474,2.988911,...,1.065159e-313,1.766927e-284,1.766921e-284,2.726806e-311,33945.0,36.571965,1,0.754515,91.749804,25612.0
4,NF0014_T1,Control,Control,Control,DMSO 1%,C4,C4-1,711.460877,936.634612,19.864556,...,3.529042e+01,3.668341e+01,3.528087e+01,3.528664e+01,50391450.0,374.135073,1,0.544161,20006.546171,27421038.0


## Perform a first round of QC by flagging any row with NaNs in metadata

We check for NaNs in the `object_id` and/or the `single_cell_count` column and flag them because:
   - An organoid can not exist if there aren't any cells.
   - NaN in object_id would be incorrect as that means the object/organoid does not exist (will have all NaNs in the feature space).

In [5]:
organoid_profiles_df = orig_organoid_profiles_df.copy()
organoid_profiles_df["Metadata_cqc_nan_detected"] = (
    organoid_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_OrganoidSingleCellCount",
            "Organoid_NoChannel_AreaSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = organoid_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

organoid_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Organoid_CenterX,Metadata_Location_Organoid_CenterY,Metadata_Location_Organoid_CenterZ,...,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_EquivalentDiameter,Organoid_NoChannel_AreaSizeShape_EulerNumber,Organoid_NoChannel_AreaSizeShape_Extent,Organoid_NoChannel_AreaSizeShape_SurfaceArea,Organoid_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.969913,393.226576,3.055781,...,1.766927e-284,0.000000e+00,1.766927e-284,72576.0,46.946452,1,0.746473,184.733753,54176.0,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.988177,254.361071,3.051103,...,1.766927e-284,0.000000e+00,1.766927e-284,111804.0,54.388652,1,0.753470,231.971359,84241.0,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,470.988847,423.997957,3.000954,...,1.766927e-284,0.000000e+00,1.766927e-284,87230.0,50.408459,1,0.768852,148.222080,67067.0,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,416.447720,1092.972474,2.988911,...,1.766927e-284,1.766921e-284,2.726806e-311,33945.0,36.571965,1,0.754515,91.749804,25612.0,False
4,NF0014_T1,Control,Control,Control,DMSO 1%,C4,C4-1,711.460877,936.634612,19.864556,...,3.668341e+01,3.528087e+01,3.528664e+01,50391450.0,374.135073,1,0.544161,20006.546171,27421038.0,False


## Process non-NaN rows to detect abnormally small and large organoids and flag them

In [6]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in organoid_profiles_df.columns if "Metadata" in x]

In [7]:
# Process each plate (patient_id) independently in the combined dataframe

# Only process the rows that are not flagged
filtered_profile_df = organoid_profiles_df[
    ~organoid_profiles_df["Metadata_cqc_nan_detected"]
]

# Find outlier organoids based on the 'Area.Size.Shape_Organoid_VOLUME' column
print("Finding small organoid outliers...")
small_size_outliers = find_outliers(
    df=filtered_profile_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Organoid_NoChannel_AreaSizeShape_Volume": -1,  # Detect very small organoids
    },
)

# Ensure the column exists before assignment
organoid_profiles_df["Metadata_cqc_small_organoid_outlier"] = False
organoid_profiles_df.loc[
    small_size_outliers.index, "Metadata_cqc_small_organoid_outlier"
] = True

print("Finding large organoid outliers...")
large_size_outliers = find_outliers(
    df=filtered_profile_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Organoid_NoChannel_AreaSizeShape_Volume": 3,  # Detect very large organoids
    },
)

# Ensure the column exists before assignment
organoid_profiles_df["Metadata_cqc_large_organoid_outlier"] = False
organoid_profiles_df.loc[
    large_size_outliers.index, "Metadata_cqc_large_organoid_outlier"
] = True

# Update original dataframe so flags persist
organoid_profiles_df.loc[
    small_size_outliers.index, "Metadata_cqc_small_organoid_outlier"
] = True
# Print number of outliers (only in filtered rows)
small_count = filtered_profile_df.index.intersection(small_size_outliers.index).shape[0]
large_count = filtered_profile_df.index.intersection(large_size_outliers.index).shape[0]
print(f"Small organoid outliers found: {small_count}")
print(f"Large organoid outliers found: {large_count}")

# Save updated plate_df with flag columns included
output_file_path = pathlib.Path(
    f"{output_dir}/organoid_flagged_outliers.parquet"
).resolve()
organoid_profiles_df.to_parquet(output_file_path, index=False)

Finding small organoid outliers...
Number of outliers: 0 (0.00%)
Outliers Range:
Organoid_NoChannel_AreaSizeShape_Volume Min: nan
Organoid_NoChannel_AreaSizeShape_Volume Max: nan
Finding large organoid outliers...
Number of outliers: 1 (5.56%)
Outliers Range:
Organoid_NoChannel_AreaSizeShape_Volume Min: 27421038.0
Organoid_NoChannel_AreaSizeShape_Volume Max: 27421038.0
Small organoid outliers found: 0
Large organoid outliers found: 1


In [8]:
# Print example output of the flagged organoid profiles
print(organoid_profiles_df.shape)
organoid_profiles_df.head()

(18, 894)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Organoid_CenterX,Metadata_Location_Organoid_CenterY,Metadata_Location_Organoid_CenterZ,...,Organoid_Mito_Texture_Variance-3-12-256,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_EquivalentDiameter,Organoid_NoChannel_AreaSizeShape_EulerNumber,Organoid_NoChannel_AreaSizeShape_Extent,Organoid_NoChannel_AreaSizeShape_SurfaceArea,Organoid_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_small_organoid_outlier,Metadata_cqc_large_organoid_outlier
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.969913,393.226576,3.055781,...,1.766927e-284,72576.0,46.946452,1,0.746473,184.733753,54176.0,False,False,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.988177,254.361071,3.051103,...,1.766927e-284,111804.0,54.388652,1,0.753470,231.971359,84241.0,False,False,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,470.988847,423.997957,3.000954,...,1.766927e-284,87230.0,50.408459,1,0.768852,148.222080,67067.0,False,False,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,416.447720,1092.972474,2.988911,...,2.726806e-311,33945.0,36.571965,1,0.754515,91.749804,25612.0,False,False,False
4,NF0014_T1,Control,Control,Control,DMSO 1%,C4,C4-1,711.460877,936.634612,19.864556,...,3.528664e+01,50391450.0,374.135073,1,0.544161,20006.546171,27421038.0,False,False,True
